# Task 4: Sentiment Classification using TF-IDF

This notebook cleans customer reviews, compares Multinomial Naive Bayes and Logistic Regression, and predicts sentiment for new text. **Note:** The supplied dataset has only 10 reviews (5 positive, 4 negative, 1 neutral). For the requested positive/negative classifier, the single neutral review is excluded. Test results on this tiny dataset are illustrative, not reliable estimates of real-world accuracy.

In [1]:
import pandas as pd
import re
import string
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Load and explore the dataset

In [2]:
file_path = "sentiment_reviews.csv"
if not Path(file_path).exists():
    file_path = "/mnt/data/sentiment_reviews.csv"

df = pd.read_csv(file_path)
print("Dataset shape:", df.shape)
print("\nSentiment counts:\n", df["sentiment"].value_counts())
display(df.head())

Dataset shape: (10, 3)

Sentiment counts:
 sentiment
positive    5
negative    4
neutral     1
Name: count, dtype: int64


,review_id,review_text,sentiment
0,1,"The product quality is exceptional, exceeded a...",positive
1,2,Terrible customer service and the item arrived...,negative
2,3,"Average performance, decent for the price poin...",neutral
3,4,"Absolutely love this! Easy to setup, robust, a...",positive
4,5,Stopped working after two days. Extremely disa...,negative


## 2. Clean text
Lowercase, remove HTML, punctuation, and English stopwords. A lightweight regex-based tokenizer keeps the notebook easy to run without extra downloads. Optional lemmatization is not needed for this small baseline.

In [3]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    words = text.split()
    words = [word for word in words if word not in ENGLISH_STOP_WORDS]
    return " ".join(words)

df = df.dropna(subset=["review_text", "sentiment"]).copy()
df["clean_review"] = df["review_text"].apply(clean_text)
display(df[["review_text", "clean_review", "sentiment"]].head())

,review_text,clean_review,sentiment
0,"The product quality is exceptional, exceeded a...",product quality exceptional exceeded expectations,positive
1,Terrible customer service and the item arrived...,terrible customer service item arrived complet...,negative
2,"Average performance, decent for the price poin...",average performance decent price point special,neutral
3,"Absolutely love this! Easy to setup, robust, a...",absolutely love easy setup robust works flawle...,positive
4,Stopped working after two days. Extremely disa...,stopped working days extremely disappointed pu...,negative


## 3. Prepare binary labels and train/test data
Exclude neutral because this task asks for positive versus negative. Stratification keeps both classes represented in the train and test sets.

In [4]:
data = df[df["sentiment"].isin(["positive", "negative"])].copy()
X = data["clean_review"]
y = data["sentiment"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.33, random_state=42, stratify=y
)
print("Training reviews:", len(X_train))
print("Testing reviews:", len(X_test))

Training reviews: 6
Testing reviews: 3


## 4. TF-IDF and model training
Using a Pipeline ensures the TF-IDF vocabulary is learned **only from training reviews**, preventing data leakage.

In [5]:
models = {
    "Multinomial Naive Bayes": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42)
}

trained_models = {}
results = []
for name, classifier in models.items():
    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer(max_features=5000)),
        ("model", classifier)
    ])
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    accuracy = accuracy_score(y_test, predictions)
    results.append({"Model": name, "Test Accuracy": accuracy})
    trained_models[name] = pipeline
    print("\n", name)
    print("Accuracy:", round(accuracy, 3))
    print(classification_report(y_test, predictions, zero_division=0))
    print("Confusion matrix (negative, positive):\n",
          confusion_matrix(y_test, predictions, labels=["negative", "positive"]))


 Multinomial Naive Bayes
Accuracy: 0.667
              precision    recall  f1-score   support

    negative       0.50      1.00      0.67         1
    positive       1.00      0.50      0.67         2

    accuracy                           0.67         3
   macro avg       0.75      0.75      0.67         3
weighted avg       0.83      0.67      0.67         3

Confusion matrix (negative, positive):
 [[1 0]
 [1 1]]

 Logistic Regression
Accuracy: 0.333
              precision    recall  f1-score   support

    negative       0.33      1.00      0.50         1
    positive       0.00      0.00      0.00         2

    accuracy                           0.33         3
   macro avg       0.17      0.50      0.25         3
weighted avg       0.11      0.33      0.17         3

Confusion matrix (negative, positive):
 [[1 0]
 [2 0]]


## 5. Compare model accuracy

In [6]:
results_df = pd.DataFrame(results).sort_values("Test Accuracy", ascending=False)
display(results_df)
best_name = results_df.iloc[0]["Model"]
best_model = trained_models[best_name]
print("Selected model:", best_name)
print("Caution: just", len(X_test), "test examples; scores are unstable.")

,Model,Test Accuracy
0,Multinomial Naive Bayes,0.666667
1,Logistic Regression,0.333333


Selected model: Multinomial Naive Bayes
Caution: just 3 test examples; scores are unstable.


## 6. Predict new customer reviews
Use the same cleaning function and fitted TF-IDF pipeline for new text.

In [7]:
def predict_sentiment(review):
    cleaned = clean_text(review)
    if not cleaned:
        return "Please enter a meaningful review."
    return best_model.predict([cleaned])[0]

examples = [
    "Amazing quality and excellent service!",
    "Very bad experience, the product broke quickly.",
    "I love this product!"
]
for review in examples:
    print(f"{review} -> {predict_sentiment(review)}")

Amazing quality and excellent service! -> positive
Very bad experience, the product broke quickly. -> positive
I love this product! -> positive


## 7. Interactive test helper
Run this cell in Jupyter to type your own review. Leave it empty to skip.

In [8]:
# To test your own review, replace the sample text below:
user_review = "The delivery was fast and the product is excellent"
print("Review:", user_review)
print("Predicted sentiment:", predict_sentiment(user_review))

# In an interactive Jupyter session, you can also use:
# user_review = input("Enter your review: ")
# print(predict_sentiment(user_review))

Review: The delivery was fast and the product is excellent
Predicted sentiment: positive


## Conclusion
- Built a text cleaning + TF-IDF + classifier pipeline.
- Compared Multinomial Naive Bayes and Logistic Regression with held-out accuracy and classification reports.
- Added a `predict_sentiment()` helper.
- **Limitation:** Nine labeled binary reviews are far too few for trustworthy evaluation. Collect a substantially larger balanced dataset before using this classifier for real decisions. The neutral example was excluded rather than silently relabeled.